# Agentic AI and RAGs ?

Using only open source frameworks.
- Tiny KB with FAISS + FakeEmbeddings
- Free Wikipedia utility (no API keys)
- Rule-based planner
- Stub summarizer with optional tiny HF model (sshleifer/tiny-gpt2)
Run all cells top to bottom in Colab.

In [2]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## 1) Build the KB retriever

In [3]:
# ===== 1) Build the KB retriever =====

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

kb_docs = [
    Document(
        page_content="""
Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.
Their core loop is: (1) interpret the user goal, (2) inspect available context, (3) decide whether tools are needed,
(4) call one or more tools in a planned sequence, and (5) synthesize an answer grounded in the tool results.
""".strip(),
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base and are the primary interface to internal documents.
Given a user query, the system should retrieve top-k candidates and base the answer primarily on those passages.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a broad-coverage, free fallback when the curated knowledge base lacks coverage or appears incomplete.
It should not be the first resort when high-quality internal documents exist.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin or conflicting, the system must be transparent about uncertainty instead of fabricating details.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Answers should be concise, well-structured, and explicitly cite the sources that support key claims.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

embeddings = FakeEmbeddings(size=256)

# Create FAISS vector store from documents
vs = FAISS.from_documents(kb_docs, embeddings)

# Build retriever
retriever = vs.as_retriever(search_kwargs={"k": 3})

print("KB ready with", len(kb_docs), "docs")


KB ready with 5 docs


## 2) Open source external tool: Wikipedia search

In [4]:
# ===== 2) Wikipedia external tool =====

from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaAPIWrapper(
    lang="en",
    top_k_results=2,
    doc_content_chars_max=1000,
)

def wiki_search(query: str, k: int = 2):
    """
    Search Wikipedia and return short snippets with titles.
    """
    try:
        results = wiki.run(query)
        snippets = []
        for r in results[:k]:
            snippets.append({
                "title": r["title"],
                "summary": r["summary"],
            })
        return snippets, None
    except Exception as e:
        return [], str(e)

# Quick sanity check
print(wiki_search("Python programming")[0][:1])


[]


## 3) Simple planner (rule-based)

In [5]:
# ===== 3) Simple rule-based planner =====

kb_keywords = [
    "agentic",
    "retriever",
    "retrieval",
    "citation",
    "ground",
    "honest",
    "transparen",
]

def plan(question: str):
    """
    Decide whether to use KB retrieval or Wikipedia based on keywords.
    """
    q = question.lower()
    for kw in kb_keywords:
        if kw in q:
            return {"action": "kb"}
    return {"action": "wiki"}

# Tests
print(plan("How to ground answers?"))
print(plan("Who created Python?"))


{'action': 'kb'}
{'action': 'wiki'}


## 4) Answer function with stub or tiny HF model

In [6]:
# ===== 4) Answer function =====

from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

prompt = ChatPromptTemplate.from_template(
    "You are a helpful agentic assistant.\n"
    "Use the given context and wiki snippets.\n"
    "If evidence is weak, say so and suggest a follow-up.\n"
    "Cite sources like [kb:doc] or [wiki:Title].\n\n"
    "Question: {question}\n"
    "Context:\n{context}\n\n"
    "Wiki:\n{wiki}\n\n"
    "Answer:"
)

def get_tiny_generator(model_id: str = "sshleifer/tiny-gpt2"):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    return pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        torch_dtype=torch.float32,
        device_map="auto",
    )

def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 120):
    gen = get_tiny_generator()
    out = gen(prompt_text, max_new_tokens=max_new_tokens)
    full = out[0]["generated_text"]
    return full[len(prompt_text):].strip()

def answer_question(question: str, use_tiny_model: bool = True):
    pl = plan(question)

    docs = retriever.invoke(question) if pl["action"] == "kb" else []
    wiki_snips, wiki_err = ([], None)

    if pl["action"] == "wiki":
        wiki_snips, wiki_err = wiki_search(question)

    context_text = "\n".join(
        [f"[{d.metadata.get('source')}] {d.page_content}" for d in docs]
    ) or "No KB context."

    wiki_text = "\n".join(
        [f"[wiki:{s['title']}] {s['summary']}" for s in wiki_snips]
    ) or "No wiki snippets."

    messages = prompt.format_messages(
        question=question,
        context=context_text,
        wiki=wiki_text,
    )

    if use_tiny_model:
        final_answer = summarize_with_tiny(messages[0].content)
    else:
        stub = FakeListChatModel(
            responses=["Stub answer based on provided context and wiki snippets."]
        )
        final_answer = stub.invoke(messages).content

    return {
        "plan": pl,
        "kb_sources": [d.metadata.get("source") for d in docs],
        "wiki_sources": [s.get("title") for s in wiki_snips],
        "wiki_error": wiki_err,
        "answer": final_answer,
    }


## 5) Quick check on sample questions

In [7]:
# ===== 5) Quick check =====

tests = [
    "What is an agentic AI system?",
    "Who created Python?",
    "How should an AI behave when evidence is missing?",
]

for q in tests:
    res = answer_question(q, use_tiny_model=False)
    print("\nQ:", q)
    print("Plan:", res["plan"])
    print("KB sources:", res["kb_sources"])
    print("Wiki sources:", res["wiki_sources"])
    print("Answer:", res["answer"])



Q: What is an agentic AI system?
Plan: {'action': 'kb'}
KB sources: ['kb:agentic_concept', 'kb:style', 'kb:wikipedia_tip']
Wiki sources: []
Answer: Stub answer based on provided context and wiki snippets.

Q: Who created Python?
Plan: {'action': 'wiki'}
KB sources: []
Wiki sources: []
Answer: Stub answer based on provided context and wiki snippets.

Q: How should an AI behave when evidence is missing?
Plan: {'action': 'wiki'}
KB sources: []
Wiki sources: []
Answer: Stub answer based on provided context and wiki snippets.
